# Stage 1.0 – EDA: Fetch Best Sources

This notebook attempts to download PDF versions of the listed clinical nutrition and medical texts.  
It saves all successfully fetched files into `./stage1_0_eda_best_sources/`.  
If a file cannot be obtained after `TRY_FETCH_FILE_NOT_MORE_THAN` attempts, its title is written to `./stage1_0_eda_best_sources/not_found_sources.txt`.

In [1]:
import os
import re
import time
import requests
from pathlib import Path
from typing import List, Dict, Optional

# ----------------------------------------------------------------------
# Configuration
# ----------------------------------------------------------------------
OUTPUT_DIR = Path("./stage1_0_eda_best_sources")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TRY_FETCH_FILE_NOT_MORE_THAN = 5   # maximum attempts per book
REQUEST_TIMEOUT = 60                 # seconds per request
RETRY_DELAY = 2                      # seconds between attempts

# Google AI API key (provided by the user) – kept for future use.
from dotenv import load_dotenv
import os
load_dotenv()
GOOGLE_AI_API_KEY = os.getenv("GOOGLE_AI_API_KEY", "")


## Book definitions

Each book is described by a display title and a list of candidate URLs.  
The program will try each URL in order, and if a URL fails it will move to the next one.  
After exhausting the list it will start over, up to `TRY_FETCH_FILE_NOT_MORE_THAN` total attempts per book.

In [2]:
BOOKS: List[Dict] = [
    {
        "title": "Krause's Food & the Nutrition Care Process",
        "urls": [
            "http://ndl.ethernet.edu.et/bitstream/123456789/27775/1/4.pdf",
            "https://digrep.mchs.mw/items/a7a74129-6343-4728-9eb5-b4bceb2281b8/full",
        ],
    },
    {
        "title": "Advanced Nutrition and Human Metabolism",
        "urls": [
            "https://www.ndl.ethernet.edu.et/bitstream/123456789/10968/1/Sareen%20S.%20Gropper%2c.pdf",
        ],
    },
    {
        "title": "Nutrition Therapy and Pathophysiology",
        "urls": [
            "https://vdoc.pub/download/nutrition-therapy-and-pathophysiology-2nd-edition-1hpfbo300g0o",
        ],
    },
    {
        "title": "Medical Nutrition and Disease: A Case-Based Approach",
        "urls": [
            "https://vdoc.pub/download/medical-nutrition-and-disease-a-case-based-approach-1c9bkpn7p3f0",
        ],
    },
    {
        "title": "The Nutrition Society Textbook",
        "urls": [
            "https://annas-archive.gd/md5/fc17a30ac23da4898257f386d6317a4c",
        ],
    },
    {
        "title": "Sports Nutrition: A Handbook for Professionals",
        "urls": [
            "https://archive.org/download/sportsnutritionp0000unse/sportsnutritionp0000unse.pdf",
        ],
    },
    {
        "title": "Motivational Interviewing in Nutrition and Fitness",
        "urls": [
            "https://vdoc.pub/download/motivational-interviewing-in-nutrition-and-fitness-6m3qfs3ruo50",
            "https://annas-archive.gd/md5/ebc10bf38b37385ba3e0c77d4d26e20e",
        ],
    },
    {
        "title": "Intuitive Eating: A Revolutionary Program that Works",
        "urls": [
            "https://vdoc.pub/download/intuitive-eating-a-revolutionary-program-that-works-5vmiruf15me0",
            "https://annas-archive.gd/md5/0ced77d1541577a2808d4437ac6f26e8",
        ],
    },
]

## Helper functions

In [3]:
def sanitize_filename(title: str) -> str:
    """Turn a book title into a safe filename."""
    # Replace any character that is not a letter, digit, space, dash, or underscore
    safe = re.sub(r'[^\w\s-]', '', title)
    safe = safe.strip().replace(' ', '_')
    return safe + ".pdf"


def download_file(url: str, dest: Path, timeout: int = REQUEST_TIMEOUT) -> bool:
    """
    Download a file from *url* to *dest*.
    Returns True on success, False otherwise.
    """
    try:
        headers = {
            "User-Agent": (
                "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                "AppleWebKit/537.36 (KHTML, like Gecko) "
                "Chrome/120.0.0.0 Safari/537.36"
            )
        }
        with requests.get(url, stream=True, timeout=timeout, headers=headers) as r:
            r.raise_for_status()
            # Check content type – we expect a PDF
            content_type = r.headers.get("Content-Type", "").lower()
            if "pdf" not in content_type and "octet-stream" not in content_type:
                # Some servers mis-report; we still try to save
                pass
            with open(dest, "wb") as f:
                for chunk in r.iter_content(chunk_size=8192):
                    if chunk:
                        f.write(chunk)
        # Basic sanity check: file must be larger than 1 KB
        if dest.stat().st_size < 1024:
            dest.unlink(missing_ok=True)
            return False
        return True
    except Exception as exc:
        # Clean up partial file if it exists
        if dest.exists():
            dest.unlink(missing_ok=True)
        print(f"  Download error: {exc}")
        return False

## Main fetching loop

For each book we cycle through the candidate URLs.  
We stop as soon as a download succeeds, or when we have made `TRY_FETCH_FILE_NOT_MORE_THAN` total attempts.

In [4]:
ot_found: List[str] = []

for book in BOOKS:
    title = book["title"]
    urls = book["urls"]
    filename = sanitize_filename(title)
    dest_path = OUTPUT_DIR / filename

    # Skip if already downloaded
    if dest_path.exists():
        print(f"[✓] Already present: {filename}")
        continue

    print(f"\n[ ] Fetching: {title}")
    attempt = 0
    success = False

    while attempt < TRY_FETCH_FILE_NOT_MORE_THAN and not success:
        url = urls[attempt % len(urls)]   # cycle through URLs
        attempt += 1
        print(f"  Attempt {attempt}/{TRY_FETCH_FILE_NOT_MORE_THAN} – {url}")

        if download_file(url, dest_path):
            print(f"  [✓] Saved: {dest_path}")
            success = True
        else:
            time.sleep(RETRY_DELAY)

    if not success:
        not_found.append(title)
        print(f"  [✗] Failed after {attempt} attempts.")

# ----------------------------------------------------------------------
# Write report of unfetched sources
# ----------------------------------------------------------------------
report_path = OUTPUT_DIR / "not_found_sources.txt"
if not_found:
    with open(report_path, "w", encoding="utf-8") as f:
        for t in not_found:
            f.write(t + "\n")
    print(f"\n[!] Wrote {len(not_found)} missing titles to {report_path}")
else:
    # Remove stale report if everything succeeded
    report_path.unlink(missing_ok=True)
    print("\n[✓] All books fetched successfully.")

[✓] Already present: Krauses_Food__the_Nutrition_Care_Process.pdf
[✓] Already present: Advanced_Nutrition_and_Human_Metabolism.pdf

[ ] Fetching: Nutrition Therapy and Pathophysiology
  Attempt 1/5 – https://vdoc.pub/download/nutrition-therapy-and-pathophysiology-2nd-edition-1hpfbo300g0o
  Download error: 403 Client Error: Forbidden for url: https://vdoc.pub/download/nutrition-therapy-and-pathophysiology-2nd-edition-1hpfbo300g0o
  Attempt 2/5 – https://vdoc.pub/download/nutrition-therapy-and-pathophysiology-2nd-edition-1hpfbo300g0o
  Download error: 403 Client Error: Forbidden for url: https://vdoc.pub/download/nutrition-therapy-and-pathophysiology-2nd-edition-1hpfbo300g0o
  Attempt 3/5 – https://vdoc.pub/download/nutrition-therapy-and-pathophysiology-2nd-edition-1hpfbo300g0o
  Download error: 403 Client Error: Forbidden for url: https://vdoc.pub/download/nutrition-therapy-and-pathophysiology-2nd-edition-1hpfbo300g0o
  Attempt 4/5 – https://vdoc.pub/download/nutrition-therapy-and-patho

## Notes

* The `GOOGLE_AI_API_KEY` is stored but not used in this notebook.  
  It can be employed in later stages for semantic search, summarisation, or fallback source discovery.
* Some URLs (e.g. Anna's Archive, vdoc.pub) may require authentication or may change structure.  
  The retry loop will keep trying, but if a site is persistently unavailable the book will appear in `not_found_sources.txt`.
* For Internet Archive items that are borrow‑only, the direct `.pdf` URL will not work;  
  you may need to use the `archive.org` lending API or manually download via a browser.